#  EHR Data Cleaning — Part 2: Noise Removal & Text Cleaning

In this notebook, you'll clean the EHR data by removing various types of noise and preparing text for extraction.

---

##  What You'll Do Today

1. **Remove duplicate patient records** — Find and eliminate duplicate rows in the patients table

2. **Remove orphaned notes** — Identify clinical notes referencing non-existent patients

3. **Remove empty and placeholder notes** — Clean out notes with no useful content

4. **Strip HTML artifacts** — Remove HTML tags that contaminate note text

5. **Fix OCR errors** — Correct character substitution errors in clinical notes

---

##  Why Data Cleaning Matters

Real-world EHR data contains many quality issues:
- **Duplicate records** from system migrations or data entry errors
- **Orphaned records** that reference deleted patients
- **Empty entries** from incomplete documentation
- **Technical artifacts** like HTML tags from copy-paste
- **OCR errors** from scanned document digitization

Cleaning these issues is essential before any analysis or ML model building.

## Setting Environment Up for Colab

Mount Google Drive and download the dataset so this notebook can access the EHR data.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Set your Google Drive path here
# Replace <PATH_TO_REPO> with your actual repository path
REPO_PATH = "/content/drive/MyDrive/<PATH_TO_REPO>"

# Data directory (source data)
DATA_DIR = f"{REPO_PATH}/week_1/data"
NOTES_PATH = f"{DATA_DIR}/notes_for_extraction.csv"

# Output directory (cleaned data)
OUTPUT_DIR = f"{REPO_PATH}/week_1/cleaned_data"

print(f"Data directory: {DATA_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

---

##  Step 1: Import Libraries

We need several Python libraries for data cleaning:

| Library | Purpose |
|---------|---------|
| **pandas** | Load and manipulate CSV data |
| **numpy** | Handle missing values and numerical operations |
| **re** | Regular expressions for pattern matching (HTML tags, OCR errors) |
| **os** | File path operations for cross-platform compatibility |
| **json** | Parse JSON data if needed |
| **Counter** | Count word frequencies for OCR analysis |

These are standard data processing libraries that form the foundation of any ETL (Extract-Transform-Load) pipeline.

In [ ]:
import pandas as pd
import numpy as np
import re
import os
import json
from collections import Counter

print("[OK] Libraries imported successfully")

---

##  Step 2: Load the Source Data

We'll load the source data, which contains various **data quality issues** for you to identify and clean:

**CSV Tables (7 files):**
- `patients.csv` — Contains duplicate records
- `encounters.csv`, `conditions.csv`, `medications.csv`, `observations.csv`, `procedures.csv`, `allergies.csv` — Standard clinical tables

**Clinical Notes:**
- `notes_for_extraction.csv` — Contains orphaned notes, empty notes, placeholder text, HTML artifacts, and OCR errors

This simulates real-world EHR data that has accumulated quality issues over time from system migrations, scanning paper records, and human data entry errors.

In [ ]:
# Load data from Google Drive
# Load CSV tables from csv/ subfolder
df_patients = pd.read_csv(os.path.join(DATA_DIR, "csv/patients.csv"))
df_encounters = pd.read_csv(os.path.join(DATA_DIR, "csv/encounters.csv"))
df_conditions = pd.read_csv(os.path.join(DATA_DIR, "csv/conditions.csv"))
df_medications = pd.read_csv(os.path.join(DATA_DIR, "csv/medications.csv"))
df_observations = pd.read_csv(os.path.join(DATA_DIR, "csv/observations.csv"))
df_procedures = pd.read_csv(os.path.join(DATA_DIR, "csv/procedures.csv"))
df_allergies = pd.read_csv(os.path.join(DATA_DIR, "csv/allergies.csv"))

# Load clinical notes (in main data folder, not csv subfolder)
df_notes = pd.read_csv(NOTES_PATH)

# Get column name for notes
note_col = "note_text"

print("All data loaded")
print(f"\nInitial counts:")
print(f"  Patients: {len(df_patients):,}")
print(f"  Notes: {len(df_notes):,}")

###  Helper Function: Case-Insensitive Column Lookup

One common challenge with real-world data is **inconsistent column naming**. The Synthea dataset uses uppercase column names (`ID`, `PATIENT`), but other datasets might use lowercase. To handle this gracefully, we create a helper function that finds columns regardless of case.

We also need to identify which column contains the actual note text — different exports may name it differently (`noisy_note`, `note_text`, `original_note`).

In [ ]:
# Import helper function from Part 1
import sys
sys.path.insert(0, f'{REPO_PATH}/src')
from helpers import get_col

print("get_col imported from src/helpers.py")

---

#  SECTION 1: Data Noise Removal

In this section, we'll systematically identify and remove 5 types of data quality issues. Each type requires a different detection and cleaning strategy:

| Issue | Detection Method | Action |
|-------|------------------|--------|
| **Duplicate patients** | Check for repeated patient IDs | Remove duplicates (keep first) |
| **Orphaned notes** | Patient ID not in patients table | Remove entire record |
| **Empty notes** | Null, empty string, or whitespace only | Remove entire record |
| **Placeholder notes** | Suspiciously short length | Remove entire record |
| **HTML artifacts** | Regex pattern for `<tags>` | Strip tags but keep note |

**Key Distinction:** For the first 4 issues, we **remove the entire record**. For HTML artifacts, we **keep the note** but clean the text — the underlying clinical content is still valid.

Let's tackle each issue one by one.

---

## Step 1: Duplicate Patient Detection & Removal

**The Problem:** Duplicate patient records create serious issues in healthcare analytics:
- **Inflated statistics**: Patient counts and encounter rates become artificially high
- **Broken joins**: When joining tables, duplicates create unexpected row multiplication
- **Incorrect risk scores**: ML models may see the same patient twice with different feature values

**How Duplicates Occur:**
- System migrations where patients were imported multiple times
- Data entry errors creating multiple records for the same person
- ETL pipeline failures causing re-insertion of batches

**Our Strategy:** Identify rows with duplicate patient IDs and keep only the first occurrence. We use `keep='first'` because the first record is typically the original, and later ones are accidental copies.

Let's examine the duplicate patients more closely to understand the scope of the problem:

In [ ]:
# Find duplicate patients by ID
patient_id_col = get_col(df_patients, 'id')
print(f"Patient ID column: '{patient_id_col}'")

print(f"\nOriginal patient count: {len(df_patients):,}")

# Find all duplicated rows (keep=False marks ALL duplicates, not just the extras)
duplicate_mask = df_patients.duplicated(subset=[patient_id_col], keep=False)
duplicates = df_patients[duplicate_mask]

print(f"Rows involved in duplication: {len(duplicates):,}")
print(f"Unique IDs with duplicates: {duplicates[patient_id_col].nunique():,}")

Now let's remove the duplicates, keeping only the first occurrence of each patient ID:

In [ ]:
# Show sample of duplicates
print("Sample duplicated patient IDs:")
sample_dup_ids = duplicates[patient_id_col].value_counts().head(3).index.tolist()
for pid in sample_dup_ids:
    count = len(df_patients[df_patients[patient_id_col] == pid])
    print(f"  {pid[:20]}... appears {count} times")

In [ ]:
# Remove duplicates - keep first occurrence
df_patients_clean = df_patients.drop_duplicates(subset=[patient_id_col], keep='first')

removed_count = len(df_patients) - len(df_patients_clean)
print(f"[OK] Removed {removed_count:,} duplicate patient rows")
print(f"   Clean patient count: {len(df_patients_clean):,}")

---

## Step 2: Orphaned Notes Detection & Removal

**The Problem:** Orphaned notes reference a `patient_id` that doesn't exist in the patients table. These notes are "orphans" — they have no parent patient record to link to.

**Why This Matters:**
- These notes **cannot be used** for patient-level analysis or ML features
- Attempting to join orphaned notes will result in `NaN` patient demographics
- They represent data integrity violations in the relational model

**How Orphans Occur:**
- Patient records were deleted but associated notes remained
- Data import errors with mismatched or corrupted IDs
- System integration failures between EHR modules

**Our Strategy:** 
1. Create a set of valid patient IDs from the cleaned patients table
2. Check each note's `patient_id` against this valid set
3. Remove any note where the patient_id is not found

This is a classic **referential integrity check** — ensuring foreign keys point to valid primary keys.

Let's examine some of the orphaned patient IDs to confirm they don't exist in our patients table:

In [ ]:
# Get set of valid patient IDs from cleaned patients table
valid_patient_ids = set(df_patients_clean[patient_id_col].astype(str).tolist())
print(f"Valid patient IDs: {len(valid_patient_ids):,}")

# Check which notes have invalid patient_id
df_notes['patient_id_str'] = df_notes['patient_id'].astype(str)
orphaned_mask = ~df_notes['patient_id_str'].isin(valid_patient_ids)

print(f"\nTotal notes: {len(df_notes):,}")
print(f"Orphaned notes (invalid patient_id): {orphaned_mask.sum():,}")

Remove all orphaned notes from our dataset:

In [ ]:
# Show sample of orphaned notes
print("Sample orphaned patient IDs:")
orphaned_sample = df_notes[orphaned_mask]['patient_id'].head(5).tolist()
for pid in orphaned_sample:
    print(f"  {pid}")

In [ ]:
# Remove orphaned notes
df_notes_clean = df_notes[~orphaned_mask].copy()

removed_count = orphaned_mask.sum()
print(f"[OK] Removed {removed_count:,} orphaned notes")
print(f"   Remaining notes: {len(df_notes_clean):,}")

---

## Step 3: Empty Notes Detection & Removal

**The Problem:** Empty notes contain no clinical information whatsoever. They're essentially blank rows that provide no value for analysis or ML.

**What Counts as "Empty":**
- `NULL` / `NaN` values — the note field was never populated
- Empty string `""` — a note was created but no text was entered
- Whitespace only `"   "` — contains only spaces, tabs, or newlines

**Why Empty Notes Exist:**
- Placeholder records created by the EHR system but never completed
- Notes where the text was accidentally deleted during editing
- System-generated skeleton records from automated workflows

**Our Strategy:** Use a compound boolean condition to catch all three cases:
1. `isna()` — catches NULL/NaN
2. `str.strip() == ''` — catches empty strings and whitespace-only

Unlike HTML artifacts, empty notes have **no recoverable content** and must be removed entirely.

Remove the empty notes from our dataset:

In [ ]:
# Find empty notes
empty_mask = (
    df_notes_clean[note_col].isna() | 
    (df_notes_clean[note_col].astype(str).str.strip() == '')
)

print(f"Empty notes found: {empty_mask.sum():,}")

In [ ]:
# Remove empty notes
df_notes_clean = df_notes_clean[~empty_mask].copy()

removed_count = empty_mask.sum()
print(f"[OK] Removed {removed_count:,} empty notes")
print(f"   Remaining notes: {len(df_notes_clean):,}")

---

## Step 4: Short/Placeholder Notes Detection & Removal

**The Problem:** Some notes pass the "empty" check but are still too short to contain meaningful clinical content. These are often:
- **Placeholder text** like `"..."`, `"TBD"`, `"pending"`, `"[note]"`
- **Incomplete entries** where a clinician started but never finished
- **System artifacts** from automated note generation

**Discovery Approach:** Rather than defining a list of known placeholders, we'll:
1. Calculate the length of each note
2. Examine the distribution to find suspiciously short notes
3. Inspect the actual content to confirm they're not useful
4. Set a reasonable length threshold to filter them out

**Why This Matters:** A real clinical note should contain at minimum:
- Patient identifier
- Date/time
- Some clinical content (chief complaint, assessment, etc.)

This typically requires at least 50-100 characters. Notes shorter than this are almost certainly incomplete or placeholder text.

In [ ]:
# Calculate note lengths
df_notes_clean['note_length'] = df_notes_clean[note_col].astype(str).str.len()

print("Note Length Statistics:")
print(f"  Min length: {df_notes_clean['note_length'].min()}")
print(f"  Max length: {df_notes_clean['note_length'].max():,}")
print(f"  Mean length: {df_notes_clean['note_length'].mean():.0f}")
print(f"  Median length: {df_notes_clean['note_length'].median():.0f}")

# Look at distribution of short notes
print(f"\nNotes by length:")
print(f"  < 20 chars:  {(df_notes_clean['note_length'] < 20).sum():,}")
print(f"  < 50 chars:  {(df_notes_clean['note_length'] < 50).sum():,}")
print(f"  < 100 chars: {(df_notes_clean['note_length'] < 100).sum():,}")

First, let's examine the length distribution to understand what "short" means in this dataset:

In [ ]:
# Let's look at what the shortest notes actually contain
print("Shortest notes in the dataset:")
print("=" * 50)

shortest_notes = df_notes_clean.nsmallest(20, 'note_length')[[note_col, 'note_length']]

for idx, row in shortest_notes.iterrows():
    print(f"[{row['note_length']:3d} chars] \"{row[note_col]}\"")

print("\n[!] Notice: These short notes are placeholder text, not real clinical content!")

Let's inspect the actual content of the shortest notes to confirm they're placeholders:

In [ ]:
# Remove notes that are too short to contain meaningful clinical content
# A real clinical note should have at least ~50 characters (patient info, date, some content)

LENGTH_THRESHOLD = 50
short_mask = df_notes_clean['note_length'] < LENGTH_THRESHOLD

print(f"Notes shorter than {LENGTH_THRESHOLD} characters: {short_mask.sum():,}")

# Remove short notes
df_notes_clean = df_notes_clean[~short_mask].copy()

# Clean up the helper column
df_notes_clean = df_notes_clean.drop(columns=['note_length'])

print(f"[OK] Removed {short_mask.sum():,} short/placeholder notes")
print(f"   Remaining notes: {len(df_notes_clean):,}")

Based on our analysis, notes under 50 characters are placeholders. Let's remove them:

---

## Step 5: HTML Artifact Detection & Cleaning

**The Problem:** HTML tags appear in clinical notes when text is copy-pasted from web interfaces or EHR systems that store data in HTML format internally.

**Common HTML Artifacts:**
```html
<div>Note text here</div>
<p>Paragraph content</p>
<br> or <br/>  (line breaks)
<span style="...">Styled text</span>
```

**Key Difference from Other Noise:** 
- Unlike duplicates, orphans, empty, and placeholder notes — **the underlying content is valid**
- We don't want to delete these notes; we want to **clean them**
- The clinical information is intact, just wrapped in HTML tags

**Our Strategy:**
1. Detect notes containing HTML using regex pattern `<[^>]+>` (matches any `<tag>`)
2. Strip all HTML tags while preserving the text content
3. Keep the cleaned note in our dataset

**Why HTML Gets Into Notes:**
- Copy-paste from web-based EHR interfaces
- Data exports that preserve internal formatting
- Migration from systems that used HTML storage

In [ ]:
# Detect HTML tags
html_pattern = r'<[^>]+>'

html_mask = df_notes_clean[note_col].str.contains(html_pattern, regex=True, na=False)

print(f"Notes with HTML artifacts: {html_mask.sum():,}")

First, let's detect how many notes contain HTML tags using a regex pattern:

In [ ]:
# Show sample of HTML-contaminated notes
print("Sample HTML artifacts:")
html_samples = df_notes_clean[html_mask][note_col].head(3)
for i, note in enumerate(html_samples):
    # Show just the first 100 chars to see the HTML
    preview = str(note)[:100]
    print(f"  {i+1}. {preview}...")

Let's preview some HTML-contaminated notes to see what we're dealing with:

In [ ]:
# Strip HTML tags from note text
def strip_html(text):
    """Remove HTML tags from text."""
    if pd.isna(text):
        return text
    return re.sub(html_pattern, '', str(text))

# Apply HTML stripping
df_notes_clean[note_col] = df_notes_clean[note_col].apply(strip_html)

print(f"[OK] Stripped HTML from {html_mask.sum():,} notes")

Now let's create a function to strip HTML tags and apply it to all notes:

In [ ]:
# Verify HTML was removed
html_remaining = df_notes_clean[note_col].str.contains(html_pattern, regex=True, na=False).sum()
print(f"Notes still containing HTML: {html_remaining}")

---

##  Noise Removal Summary

Before moving to OCR correction, let's summarize what we've cleaned so far. This checkpoint helps us:
- Verify each step removed the expected number of records
- Confirm we haven't accidentally removed too much data
- Document the cleaning pipeline for reproducibility

In [ ]:
print("=" * 70)
print("NOISE REMOVAL SUMMARY")
print("=" * 70)

print(f"\nPATIENTS:")
print(f"  Original: {len(df_patients):,}")
print(f"  After removing duplicates: {len(df_patients_clean):,}")
print(f"  Removed: {len(df_patients) - len(df_patients_clean):,}")

print(f"\nNOTES:")
print(f"  Original: {len(df_notes):,}")
print(f"  After cleaning: {len(df_notes_clean):,}")
print(f"  Total removed: {len(df_notes) - len(df_notes_clean):,}")

print(f"\n" + "=" * 70)
print("[OK] Data noise removal complete!")
print("=" * 70)

---

#  SECTION 2: OCR Error Correction

**The Problem:** When paper medical records are digitized using Optical Character Recognition (OCR), the scanning software often confuses visually similar characters. This creates "mixed" words where some letters have been replaced by look-alike digits (or vice versa).

**Common OCR Confusions:**
| Digit | Often Misread As | Visual Similarity |
|-------|------------------|-------------------|
| `0` | `O` or `o` | Circular shapes |
| `1` | `l` or `I` | Vertical lines | 
| `3` | `E` or `e` | Mirrored curves |
| `5` | `S` or `s` | Curved top |
| `8` | `B` or `b` | Double loops |

**Real Examples You'll See:**
- `diab3tes` → should be `diabetes`
- `med1cation` → should be `medication`  
- `m0nitor` → should be `monitor`
- `pati3nt` → should be `patient`

**Why This Matters for NLP:**
- Word embeddings won't recognize `diab3tes` as related to `diabetes`
- Keyword searches will miss OCR-corrupted terms
- Clinical entity recognition will fail on corrupted drug names

**Our Approach:** We'll use two complementary techniques:
1. **Data-driven analysis** — Discover systematic patterns from word frequencies
2. **Rule-based correction** — Apply digit-to-letter substitutions to mixed words

###  Step 1: Build Vocabulary and Identify Suspicious Words

To find OCR errors, we first need to understand the vocabulary of our clinical notes. We'll:
1. **Tokenize** all notes into individual words
2. **Count word frequencies** to understand what's common vs rare
3. **Identify mixed words** — words containing both letters AND digits

Words that mix letters and digits are strong candidates for OCR errors (unless they're valid medical terms like `HbA1c` or `COVID-19`).

In [ ]:
# Build vocabulary from all notes
all_text = ' '.join(df_notes_clean[note_col].dropna().astype(str).tolist())

# Tokenize into words
words = re.findall(r'\b[a-zA-Z0-9]+\b', all_text.lower())
word_freq = Counter(words)

print(f"Total words: {len(words):,}")
print(f"Unique words: {len(word_freq):,}")

Now let's identify words that mix letters and digits — these are our OCR error candidates:

In [ ]:
# Find words that mix letters and digits (potential OCR errors)
def has_letter_and_digit(word):
    """Check if word contains both letters and digits."""
    has_letter = any(c.isalpha() for c in word)
    has_digit = any(c.isdigit() for c in word)
    return has_letter and has_digit

# Known valid terms with digits (should not be corrected)
valid_mixed_terms = {
    'covid19', 'covid-19', 'sars-cov-2', 'h1n1', 'b12', 'd3',
    'a1c', 'hba1c', 'type1', 'type2', 't1dm', 't2dm',
    'mg', 'ml', 'g', 'kg', 'mcg', 'iu',
    '24hr', '12hr', '2x', '3x', '4x'
}

# Find suspicious mixed words
mixed_words = {
    word: freq for word, freq in word_freq.items()
    if has_letter_and_digit(word) and word.lower() not in valid_mixed_terms
}

print(f"Mixed letter-digit words: {len(mixed_words):,}")
print(f"\nTop 20 suspicious mixed words:")
for word, freq in sorted(mixed_words.items(), key=lambda x: -x[1])[:20]:
    print(f"  {word}: {freq}")

###  Step 2: Data-Driven OCR Analysis with Edit Distance

Instead of manually guessing which characters are confused, we can **discover patterns from the data** using a technique called **edit distance matching**.

**The Idea:**
1. Split vocabulary into **low-frequency words** (likely OCR errors) and **high-frequency words** (likely correct spellings)
2. For each low-frequency word, find the **closest high-frequency word** using Levenshtein edit distance
3. Compare the character differences to extract **systematic substitution patterns**

**Why This Works:**
- OCR errors are typically **rare** (the error is random, so `diab3tes` appears less often than `diabetes`)
- Correct spellings are typically **common** (thousands of notes use the correct spelling)
- If `diab3tes` → `diabetes` with edit distance 1, and the only difference is `3` → `e`, that's evidence of a `3↔e` OCR confusion

**Levenshtein Edit Distance:** The minimum number of single-character edits (insertions, deletions, substitutions) required to transform one word into another.
- `edit_distance("diab3tes", "diabetes") = 1` (substitute `3` → `e`)
- `edit_distance("cat", "car") = 1` (substitute `t` → `r`)

First, we define parameters and split the vocabulary into low-frequency (potential errors) and high-frequency (likely correct) word sets:

In [ ]:
from collections import defaultdict

# Parameters for frequency-based analysis
LOW_MAX_FREQ = 100      # Words appearing <= this are "low frequency" (potential errors)
HIGH_MIN_FREQ = 500     # Words appearing >= this are "high frequency" (likely correct)
MIN_WORD_LEN = 4        # Minimum word length to consider
MAX_EDIT_DIST = 2       # Maximum edit distance to consider a match

# Split vocabulary into low-frequency and high-frequency sets
low_words = [w for w, c in word_freq.items() if c <= LOW_MAX_FREQ and len(w) >= MIN_WORD_LEN]
high_words = [w for w, c in word_freq.items() if c >= HIGH_MIN_FREQ and len(w) >= MIN_WORD_LEN]

print(f"Low-frequency words (count <= {LOW_MAX_FREQ}): {len(low_words):,}")
print(f"High-frequency words (count >= {HIGH_MIN_FREQ}): {len(high_words):,}")

In [ ]:
def edit_distance(a, b, max_dist=2):
    """
    Calculate Levenshtein distance with early termination.
    Returns max_dist+1 if distance exceeds max_dist for efficiency.
    """
    if abs(len(a) - len(b)) > max_dist:
        return max_dist + 1
    if len(a) > len(b):
        a, b = b, a
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        curr = [i]
        for j, cb in enumerate(b, 1):
            cost = 0 if ca == cb else 1
            curr.append(min(prev[j] + 1, curr[-1] + 1, prev[j-1] + cost))
        if min(curr) > max_dist:
            return max_dist + 1
        prev = curr
    return prev[-1]

# Test the function
print("Edit distance examples:")
print(f"  edit_distance('denie8', 'denies') = {edit_distance('denie8', 'denies')}")
print(f"  edit_distance('medicati0n', 'medication') = {edit_distance('medicati0n', 'medication')}")
print(f"  edit_distance('patient', 'patient') = {edit_distance('patient', 'patient')}")

Next, we implement the Levenshtein edit distance algorithm with early termination for efficiency:

In [ ]:
# Create length-based index for faster matching
len_index = defaultdict(list)
for w in high_words:
    len_index[len(w)].append(w)

# Find nearest high-frequency match for each low-frequency word
# Limit to first 1000 for computational efficiency
pairs = []
for lw in low_words[:1000]:
    L = len(lw)
    best = None
    best_d = MAX_EDIT_DIST + 1
    
    for k in range(L - MAX_EDIT_DIST, L + MAX_EDIT_DIST + 1):
        for cand in len_index.get(k, []):
            d = edit_distance(lw, cand, MAX_EDIT_DIST)
            if d < best_d:
                best, best_d = cand, d
    
    if best is not None and best_d <= MAX_EDIT_DIST:
        pairs.append((lw, word_freq[lw], best, word_freq[best], best_d))

print(f"Found {len(pairs)} potential correction pairs")

In [ ]:
# Display top correction candidates
pairs.sort(key=lambda x: (-x[3], x[0]))

print("TOP CORRECTION CANDIDATES")
print("=" * 70)
print(f"{'Low-freq word':<20} {'Freq':<6} {'High-freq word':<20} {'Freq':<6} {'Dist'}")
print("-" * 70)
for low_tok, low_cnt, high_tok, high_cnt, dist in pairs[:30]:
    print(f"{low_tok:<20} {low_cnt:<6} {high_tok:<20} {high_cnt:<6} {dist}")

Now we find the closest high-frequency match for each low-frequency word using our edit distance function:

In [ ]:
# Count character-level substitutions from our correction pairs
substitutions = Counter()

for low_tok, _, high_tok, _, _ in pairs:
    if len(low_tok) == len(high_tok):
        for c_low, c_high in zip(low_tok, high_tok):
            if c_low != c_high:
                substitutions[(c_low, c_high)] += 1

print("CHARACTER SUBSTITUTION PATTERNS")
print("=" * 50)
print(f"{'From':<8} {'To':<8} {'Count'}")
print("-" * 30)
for (src, dst), count in substitutions.most_common(20):
    print(f"{src:<8} {dst:<8} {count:>5}")

In [ ]:
# Extract digit-to-letter confusions specifically
digit_to_letter = {}
for (src, dst), count in substitutions.items():
    if src.isdigit() and dst.isalpha():
        if src not in digit_to_letter or count > digit_to_letter[src][1]:
            digit_to_letter[src] = (dst, count)

print("DIGIT TO LETTER CONFUSIONS (Data-Driven)")
print("=" * 50)
for digit, (letter, count) in sorted(digit_to_letter.items(), key=lambda x: -x[1][1]):
    print(f"  '{digit}' is often misread as '{letter}'  ({count} occurrences)")

print("\n These patterns were discovered from the data, not manually defined!")

Let's display the top correction candidates to see what patterns we've discovered:

###  Step 3: Define OCR Correction Rules

Based on common OCR error patterns and our data-driven analysis, we define a set of digit-to-letter substitution rules. When we encounter a word that mixes letters and digits (and isn't a known valid term like `HbA1c`), we'll replace each digit with its likely correct letter.

In [ ]:
# OCR digit-to-letter substitution patterns
ocr_substitutions = {
    '0': 'o',  # zero -> letter O
    '1': 'l',  # one -> lowercase L
    '3': 'e',  # three -> letter E
    '4': 'a',  # four -> letter A
    '5': 's',  # five -> letter S
    '6': 'g',  # six -> letter G
    '7': 't',  # seven -> letter T
    '8': 'b',  # eight -> letter B
}

def fix_ocr_word(word, ocr_subs, valid_terms):
    """Attempt to fix OCR errors in a word by substituting digits."""
    if not has_letter_and_digit(word):
        return word
    if word.lower() in valid_terms:
        return word
    
    # Try substituting each digit
    fixed = word.lower()
    for digit, letter in ocr_subs.items():
        fixed = fixed.replace(digit, letter)
    
    return fixed

# Test the function
test_words = ['diab3tes', 'pati3nt', 'med1cation', 'm0nitor']
print("OCR correction examples:")
for word in test_words:
    print(f"  {word} -> {fix_ocr_word(word, ocr_substitutions, valid_mixed_terms)}")

Now we extract systematic character substitution patterns from our correction pairs:

In [ ]:
def fix_ocr_text(text, ocr_subs, valid_terms):
    """Fix OCR errors in entire text."""
    if pd.isna(text):
        return text
    
    text = str(text)
    words = re.findall(r'\b[a-zA-Z0-9]+\b', text)
    
    for word in words:
        if has_letter_and_digit(word) and word.lower() not in valid_terms:
            fixed = fix_ocr_word(word, ocr_subs, valid_terms)
            if fixed != word.lower():
                # Replace preserving case of first character
                if word[0].isupper():
                    fixed = fixed.capitalize()
                text = text.replace(word, fixed)
    
    return text

print("[OK] OCR correction function defined")

In [ ]:
# Count notes with potential OCR errors before fixing
def has_ocr_errors(text, valid_terms):
    if pd.isna(text):
        return False
    words = re.findall(r'\b[a-zA-Z0-9]+\b', str(text))
    for word in words:
        if has_letter_and_digit(word) and word.lower() not in valid_terms:
            return True
    return False

ocr_error_mask = df_notes_clean[note_col].apply(lambda x: has_ocr_errors(x, valid_mixed_terms))
print(f"Notes with potential OCR errors: {ocr_error_mask.sum():,}")

###  Step 4: Apply OCR Corrections to All Notes

Now we'll apply our correction rules to every clinical note in the dataset. We'll count OCR errors before and after to measure the impact of our corrections.

In [ ]:
# Apply OCR corrections to all notes
df_notes_clean[note_col] = df_notes_clean[note_col].apply(
    lambda x: fix_ocr_text(x, ocr_substitutions, valid_mixed_terms)
)

# Verify corrections
ocr_error_mask_after = df_notes_clean[note_col].apply(lambda x: has_ocr_errors(x, valid_mixed_terms))
print(f"Notes with OCR errors after fixing: {ocr_error_mask_after.sum():,}")
print(f"\n[OK] OCR corrections applied")

---

##  Step 5: Save Cleaned Data

Now that we've completed all cleaning steps, we'll save the results to a `cleaned_data/` directory. This creates a clean separation between:

- **`data/`** — Original source data (never modify)
- **`cleaned_data/`** — Our cleaned output (reproducible from running this notebook)

**What We're Saving:**
| File | Contents | Changes Made |
|------|----------|--------------|
| `csv/patients.csv` | Patient demographics | Duplicates removed |
| `notes_for_extraction.csv` | Clinical notes | Orphans, empty, placeholders removed; HTML stripped; OCR corrected |
| `csv/*.csv` (other tables) | Clinical tables | Unchanged (no cleaning needed) |

**Best Practice:** Always save cleaned data to a new location rather than overwriting the source. This ensures you can:
1. Re-run the cleaning pipeline if you discover new issues
2. Compare cleaned vs. original data for validation
3. Track the provenance of your processed data

First, create the output directory structure:

In [ ]:
# Create output directory structure
os.makedirs(os.path.join(OUTPUT_DIR, "csv"), exist_ok=True)

print(f"[OK] Created output directory: {OUTPUT_DIR}")

---

## Important: Save Your Functions to src/

After completing this notebook, **copy your OCR functions to `src/data_processing/ocr.py`**. This makes your code reusable in the Homework and future notebooks.

**File to update:** `src/data_processing/ocr.py`

**Functions to copy:**
1. `has_letter_and_digit()` - Check for mixed letter-digit words
2. `fix_ocr_word()` - Fix OCR errors in a single word
3. `strip_html()` - Remove HTML tags from text

**Already provided in src:** `edit_distance()`, `fix_ocr_text()`, `has_ocr_errors()`

Once copied, you can import them:
```python
from data_processing.ocr import fix_ocr_text, strip_html
```

---

##  Summary: Data Cleaning Complete!

### What You Accomplished

In this notebook, you systematically cleaned EHR data by addressing 6 types of quality issues:

| Step | Issue | Technique | Result |
|------|-------|-----------|--------|
| Step 1: | Duplicate patients | `drop_duplicates()` on ID | ~100 rows removed |
| Step 2: | Orphaned notes | Referential integrity check | ~500 notes removed |
| Step 3: | Empty notes | NULL/whitespace detection | ~500 notes removed |
| Step 4: | Placeholder notes | Length threshold (<50 chars) | ~400 notes removed |
| Step 5: | HTML artifacts | Regex tag stripping | ~1,700 notes cleaned |
| Step 6: | OCR errors | Digit-to-letter substitution | ~15,000 notes corrected |

###  Key Takeaways

1. **Data cleaning is essential** before any analysis or ML — garbage in, garbage out
2. **Different noise types require different strategies** — remove vs. clean vs. transform
3. **Validation matters** — always verify your cleaning didn't remove too much data
4. **Preserve source data** — save cleaned output to a new location for reproducibility

###  Next Steps

In the **Homework**, you'll:
- Extract clinical information from the cleaned notes using regex patterns
- Identify conditions, medications, lab values, and vital signs
- Integrate extracted data with the CSV tables to build complete patient records

###  Professional Tip

Real-world data cleaning is iterative. You'll often discover new quality issues after you start analyzing the "clean" data. Build your pipeline to be re-runnable — when you find a new issue, add a cleaning step and regenerate your output. Document every transformation for auditability!

Now save all cleaned data files:

In [ ]:
# Save cleaned data
df_patients_clean.to_csv(os.path.join(OUTPUT_DIR, "csv", "patients.csv"), index=False)
df_notes_clean.to_csv(os.path.join(OUTPUT_DIR, "notes_for_extraction.csv"), index=False)

# Copy unchanged tables
df_encounters.to_csv(os.path.join(OUTPUT_DIR, "csv", "encounters.csv"), index=False)
df_conditions.to_csv(os.path.join(OUTPUT_DIR, "csv", "conditions.csv"), index=False)
df_medications.to_csv(os.path.join(OUTPUT_DIR, "csv", "medications.csv"), index=False)
df_observations.to_csv(os.path.join(OUTPUT_DIR, "csv", "observations.csv"), index=False)
df_procedures.to_csv(os.path.join(OUTPUT_DIR, "csv", "procedures.csv"), index=False)
df_allergies.to_csv(os.path.join(OUTPUT_DIR, "csv", "allergies.csv"), index=False)

print("[OK] All cleaned data saved to:", OUTPUT_DIR)
print(f"  - csv/patients.csv ({len(df_patients_clean):,} rows)")
print(f"  - notes_for_extraction.csv ({len(df_notes_clean):,} rows)")
print(f"  - csv/*.csv (6 unchanged tables)")